# Praktikum 14 – Studi Kasus Prediksi Risiko Gagal Bayar Pinjaman
**Modul Machine Learning – Bab 14 (Komparasi Model, Tuning, dan Deployment)**

Isi notebook:
1. 14.9 Contoh Perhitungan (skor gabungan F1 & interpretability)
2. 14.10 Praktikum 14.1 & 14.2 – Persiapan library, dataset, dan preprocessing
3. 14.11 Training & Tuning (Random Forest, Gradient Boosting, SVM)
4. 14.12 Evaluasi dan pemilihan model terbaik
5. 14.13 Serialisasi model (joblib)
6. 14.14 Aplikasi deployment Streamlit

> **Catatan:** modul mengasumsikan `df` sudah melalui EDA dan feature engineering. Karena data aslinya tidak tersedia, notebook ini memakai **data sintetis** dengan kolom `pendapatan`, `usia`, `lama_kerja`, dan `target` (1 = gagal bayar). Jika punya dataset sendiri, ganti sel pembuatan data dengan `df = pd.read_csv("datasetmu.csv")`.

## 14.9 Contoh Perhitungan
Skor Gabungan = 0.7 × F1-Score + 0.3 × (Interpretability / 5)

In [1]:
import pandas as pd

ringkasan = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Gradient Boosting"],
    "F1-Score (CV)": [0.81, 0.87, 0.88],
    "Interpretability (1-5)": [5, 3, 2],
})
ringkasan["Skor Gabungan"] = (0.7 * ringkasan["F1-Score (CV)"]
                              + 0.3 * ringkasan["Interpretability (1-5)"] / 5).round(3)
display(ringkasan)
print("Model terbaik menurut skor gabungan:",
      ringkasan.loc[ringkasan["Skor Gabungan"].idxmax(), "Model"])

,Model,F1-Score (CV),Interpretability (1-5),Skor Gabungan
0,Decision Tree,0.81,5,0.867
1,Random Forest,0.87,3,0.789
2,Gradient Boosting,0.88,2,0.736


Model terbaik menurut skor gabungan: Decision Tree


**Interpretasi:** Decision Tree memperoleh skor gabungan tertinggi (0.867) walaupun F1-Score-nya paling rendah (0.81), karena nilai interpretability-nya paling tinggi. Ini menunjukkan bahwa pemilihan model akhir sebaiknya mempertimbangkan lebih dari satu kriteria (akurasi dan kemudahan interpretasi), sesuai kebutuhan bisnis. Bobot 0.7/0.3 harus disepakati sesuai konteks proyek.

## 14.10 Implementasi Python
### Praktikum 14.1 Persiapan Library

In [2]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, classification_report

### Praktikum 14.2 Import Dataset dan Preprocessing
Data sintetis dibuat sebagai pengganti `df` yang di modul diasumsikan sudah melalui EDA, penanganan missing value, dan feature engineering.

In [3]:
rng = np.random.default_rng(42)
n = 2000
pendapatan = rng.normal(6_000_000, 2_500_000, n).clip(1_000_000, 25_000_000).round(-3)
usia = rng.integers(18, 70, n)
lama_kerja = (rng.gamma(2.0, 3.0, n)).clip(0, usia - 17).round(1)

# Risiko gagal bayar naik jika pendapatan rendah, usia muda, dan masa kerja pendek
logit = (-0.8
         - 0.00000045 * (pendapatan - 6_000_000)
         - 0.03 * (usia - 40)
         - 0.15 * (lama_kerja - 6)
         + rng.normal(0, 0.8, n))
target = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

df = pd.DataFrame({"pendapatan": pendapatan, "usia": usia,
                   "lama_kerja": lama_kerja, "target": target})
print(df.shape)
print(df["target"].value_counts(normalize=True).round(3))
df.head()

(2000, 4)
target
0    0.649
1    0.351
Name: proportion, dtype: float64


,pendapatan,usia,lama_kerja,target
0,6762000.0,56,1.4,0
1,3400000.0,20,3.0,1
2,7876000.0,37,20.0,0
3,8351000.0,41,13.7,0
4,1122000.0,67,6.1,1


In [4]:
# Asumsikan df sudah melalui EDA, penanganan missing value, dan feature engineering
X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (1600, 3) | Test: (400, 3)


## 14.11 Training & Tuning (Komparasi 3 Model)

In [5]:
model_configs = {
    "Random Forest": (RandomForestClassifier(random_state=42),
                      {"n_estimators": [100, 300], "max_depth": [5, 10, None]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=42),
                          {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1]}),
    "SVM": (Pipeline([("scaler", StandardScaler()),
                      ("svm", SVC(probability=True, random_state=42))]),
            {"svm__C": [1, 10], "svm__gamma": [0.01, 0.1]}),
}

In [6]:
hasil_terbaik = {}
cv_scores = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for nama, (model, params) in model_configs.items():
    search = RandomizedSearchCV(model, params, n_iter=4, cv=skf,
                                scoring="f1", random_state=42)
    search.fit(X_train, y_train)
    hasil_terbaik[nama] = search.best_estimator_
    cv_scores[nama] = search.best_score_
    print(f"{nama}: F1 CV terbaik = {search.best_score_:.3f}, params = {search.best_params_}")

Random Forest: F1 CV terbaik = 0.537, params = {'n_estimators': 100, 'max_depth': 10}


Gradient Boosting: F1 CV terbaik = 0.561, params = {'n_estimators': 200, 'learning_rate': 0.05}


SVM: F1 CV terbaik = 0.553, params = {'svm__gamma': 0.1, 'svm__C': 10}


## 14.12 Evaluasi dan Pemilihan Model Terbaik

In [7]:
f1_test = {}
for nama, model in hasil_terbaik.items():
    f1 = f1_score(y_test, model.predict(X_test))
    f1_test[nama] = f1
    print(f"{nama}: F1-Score Test = {f1:.3f}")

perbandingan = pd.DataFrame({"F1 CV": cv_scores, "F1 Test": f1_test}).round(3)
display(perbandingan)

Random Forest: F1-Score Test = 0.541
Gradient Boosting: F1-Score Test = 0.553
SVM: F1-Score Test = 0.523


,F1 CV,F1 Test
Random Forest,0.537,0.541
Gradient Boosting,0.561,0.553
SVM,0.553,0.523


In [8]:
# Pemilihan model berdasarkan F1-Score Test tertinggi (sesuaikan dengan hasil analisis Anda)
nama_terpilih = max(f1_test, key=f1_test.get)
model_terpilih = hasil_terbaik[nama_terpilih]
print("Model terpilih:", nama_terpilih)
print()
print(classification_report(y_test, model_terpilih.predict(X_test),
                            target_names=["Lancar", "Gagal bayar"]))

Model terpilih: Gradient Boosting

              precision    recall  f1-score   support

      Lancar       0.76      0.80      0.78       260
 Gagal bayar       0.59      0.52      0.55       140

    accuracy                           0.70       400
   macro avg       0.67      0.66      0.67       400
weighted avg       0.70      0.70      0.70       400



**Analisis:** Model dipilih berdasarkan F1-Score pada data uji. Pada kasus risiko kredit, *false negative* (peminjam berisiko yang lolos) umumnya lebih mahal daripada *false alarm*, sehingga F1 dan recall kelas gagal bayar perlu dipantau. Jika interpretability penting, gunakan pendekatan skor gabungan seperti pada bagian 14.9. Hasil angka dapat berbeda jika dataset asli dipakai.

## 14.13 Serialisasi Model untuk Deployment

In [9]:
joblib.dump(model_terpilih, "model_terbaik.pkl")
print("Model berhasil disimpan sebagai model_terbaik.pkl")

Model berhasil disimpan sebagai model_terbaik.pkl


## 14.14 Aplikasi Deployment Sederhana (Streamlit)
Sel di bawah menyimpan kode aplikasi ke file `app.py`. Jalankan dari terminal dengan `streamlit run app.py` (pastikan `model_terbaik.pkl` berada di folder yang sama).

In [10]:
%%writefile app.py
import streamlit as st
import joblib
import pandas as pd

model = joblib.load("model_terbaik.pkl")

st.title("Prediksi Risiko Gagal Bayar Pinjaman")
pendapatan = st.number_input("Pendapatan Bulanan (Rp)", min_value=0, value=5000000)
usia = st.number_input("Usia", min_value=18, max_value=100, value=30)
lama_kerja = st.number_input("Lama Bekerja (tahun)", min_value=0, value=3)

if st.button("Prediksi"):
    data_baru = pd.DataFrame([[pendapatan, usia, lama_kerja]],
                             columns=["pendapatan", "usia", "lama_kerja"])
    prediksi = model.predict(data_baru)[0]
    probabilitas = model.predict_proba(data_baru)[0][1]
    if prediksi == 1:
        st.error(f"Risiko TINGGI gagal bayar (probabilitas: {probabilitas:.1%})")
    else:
        st.success(f"Risiko RENDAH gagal bayar (probabilitas: {probabilitas:.1%})")

# Jalankan dengan perintah terminal: streamlit run app.py

Writing app.py


In [11]:
# Uji cepat: muat model dan prediksi satu profil contoh (tanpa Streamlit)
m = joblib.load("model_terbaik.pkl")
contoh = pd.DataFrame([[5_000_000, 30, 3]], columns=["pendapatan", "usia", "lama_kerja"])
print("Prediksi:", m.predict(contoh)[0], "| Probabilitas gagal bayar:", round(m.predict_proba(contoh)[0][1], 3))

Prediksi: 1 | Probabilitas gagal bayar: 0.677
